# Sketch2Shape on Kaggle

Runs the whole project on Kaggle's free **GPU T4 x2**: setup, tests, one example run, then the web app.

**Settings** (right panel, Session options): Accelerator **GPU T4 x2**, Internet **On**.

**Optional secrets** (Add-ons, Secrets): `ANTHROPIC_API_KEY` lets Claude name the sketched object; without it CLIP does. `HF_TOKEN` speeds up model downloads.

First run: about 20-40 minutes (install, compile, downloads). Later runs in the same session: about 1-2 minutes.

In [ ]:
# 1. Get the code. Set REPO_URL to your repository (a private repo needs a token in the URL),
#    or attach the code as a Kaggle dataset and set CODE_DATASET instead.
import os, subprocess, shutil
REPO_URL = "https://github.com/RohanSaikumar/sketch2shape.git"
CODE_DATASET = ""          # e.g. "/kaggle/input/sketch2shape-code/sketch2shape"
ROOT = "/kaggle/working/sketch2shape"

if CODE_DATASET:
    shutil.copytree(CODE_DATASET, ROOT, dirs_exist_ok=True)
elif os.path.exists(f"{ROOT}/.git"):
    print(subprocess.run(["git", "-C", ROOT, "pull"], capture_output=True, text=True).stdout)
else:
    print(subprocess.run(["git", "clone", REPO_URL, ROOT], capture_output=True, text=True))
os.chdir(ROOT)
print("working in", os.getcwd())

In [ ]:
# 2. One-time setup: Hunyuan3D-2, its CUDA extensions, a compatibility patch, our requirements
!bash scripts/setup.sh 2>&1 | tail -n 25

In [ ]:
# 3. Optional keys from Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for name in ("ANTHROPIC_API_KEY", "HF_TOKEN"):
        try:
            os.environ[name] = secrets.get_secret(name)
            print(name, "loaded")
        except Exception:
            print(name, "not set (optional)")
except ImportError:
    print("Not on Kaggle: set keys as environment variables if you want them.")

In [ ]:
# 4. Offline tests (no GPU needed): preprocessing, scoring, mesh checks, export, state machine
!python -m pytest -q tests

In [ ]:
# 5. Run every example sketch (about 3 min of model loading, then about 2-3 min per sketch).
#    Writes results/results.md, results/results.json, results/<sketch>.glb and results/sheets/<sketch>.png
#    To test your own sketches, attach them as a dataset and replace examples/sketches with its folder.
!python -u scripts/eval_batch.py examples/sketches --prompts examples/prompts.json --out results 2>&1 | grep --line-buffered -E "^==|^\{|^\||Successful|Error|error|Traceback"

In [ ]:
# 6. Look at each result: input, cleaned sketch, candidate images with scores (* = chosen), image sent to 3D
import json
from IPython.display import Markdown, display
from PIL import Image
for r in json.load(open("results/results.json")):
    display(Markdown(f"**{r['sketch']}**: object `{r['object']}` (named by {r['named_by']}), "
                     f"fidelity {r['fidelity']}, texture: {r['texture']}, {r['seconds']} s, ok={r['ok']}"))
    sheet = f"results/sheets/{r['sketch'].rsplit('.', 1)[0]}.png"
    if os.path.exists(sheet):
        display(Image.open(sheet))

In [ ]:
# 7. View the 3D models: open the printed gradio.live link, rotate each model. Stop the cell when done.
import glob
import gradio as gr
with gr.Blocks(title="Sketch2Shape results") as review:
    gr.Markdown("Sketch2Shape batch results: drag to rotate, scroll to zoom.")
    for path in sorted(glob.glob("results/*.glb")):
        with gr.Row():
            gr.Image(f"results/sheets/{os.path.basename(path)[:-4]}.png", show_label=False, scale=3)
            gr.Model3D(value=path, label=os.path.basename(path), clear_color=(1, 1, 1, 1), height=300, scale=2)
review.launch(share=True, allowed_paths=[os.path.abspath("results")])

In [ ]:
# 8. The web app for drawing live. Stop cell 7 first. Open the printed gradio.live link on any device.
!python app.py --share